# QuickCart India — Customer Behaviour and Polynomial Regression

This is my complete working notebook for the QuickCart India project. I am using the survey data about how people use quick-commerce apps for grocery shopping. I want to understand the data first, clean it properly, study it using graphs, and then check whether Linear Regression and Polynomial Regression can estimate the impulse-buying frequency score.

I am keeping everything in one notebook so I can follow the complete work from the beginning without jumping between three notebooks. I have written the explanation before every code section in the same way I would explain what I am doing while working on the project.

**One important point:** the target available in this dataset is a survey answer from 1 to 5. It is not the actual number of impulse purchases, and it is not an exact order amount in rupees. So the model estimates the reported survey score only. I will not call it a perfect prediction of a person's future shopping behaviour.

## 0. What I am going to do in this notebook

I am following these steps in order:

1. Download the original dataset directly from its public source using `urllib`.
2. Read the downloaded file in memory using `io.BytesIO`. I do not need to upload a CSV into Colab.
3. Inspect the original data and understand every column.
4. Check missing answers, duplicate rows, unusual values and the target distribution.
5. Clean the data without changing the original file.
6. Download the cleaned CSV to my computer. I will manually place it inside the project's `data` folder.
7. Explore the data through different tables and graphs. Under every graph I will write what it shows and what I can understand from it.
8. Prepare the input columns and the target column.
9. Split the data into training and testing parts before fitting preprocessing steps.
10. Compare a simple baseline, Linear Regression, Polynomial Regression and regularised Polynomial Regression.
11. Evaluate the results, inspect errors and check whether a more complex model is actually helping.
12. Save the model, metrics and useful reports. I will manually place the downloaded files in their folders.
13. Check the saved model can make a prediction for a new survey-style input.

I am using only the tools needed for this work. The main libraries are Pandas, NumPy, Matplotlib, Seaborn and Scikit-learn.

## 1. Import the libraries

Before starting, I need to import the libraries used in this notebook.

- **Pandas** helps me read and work with the table.
- **NumPy** helps with numerical calculations.
- **Matplotlib and Seaborn** help me create graphs.
- **urllib** downloads the dataset from the public source.
- **io** lets me read the downloaded file directly from memory. This means the notebook does not depend on a CSV upload from my computer.
- **Scikit-learn** provides the split, preprocessing methods, regression models and evaluation measures.

I am also setting a fixed random value wherever randomness is used, so that I can run the notebook again and get the same split.

In [ ]:
import io
import json
import math
import re
import urllib.request
import zipfile
import urllib.error
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    median_absolute_error, explained_variance_score
)
from sklearn.inspection import permutation_importance
import sklearn

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 90)
sns.set_theme(style="whitegrid", context="notebook")

print("Libraries are ready.")
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)

## 2. Download the original dataset using urllib and io

The first download address I tried was an older Mendeley API address. It now returns `404`, so I have changed the notebook to use Mendeley's current API host.

I first request the public dataset details, then find the CSV file listed for this dataset version. I download the file through `urllib` and read it from memory with `io.BytesIO`. This means I do not have to upload the CSV into Colab or save a temporary copy just to read it.

If the dataset publisher changes the file name or the public API response, this cell prints the exact step that failed. It does not quietly substitute a different dataset.


In [ ]:
# Mendeley Data's current API host is api.data.mendeley.com.
# The older data.mendeley.com/public-api/... address returns HTTP 404.
DATASET_ID = "wc2yz9485m"
DATASET_VERSION = 1
DATASET_PAGE = "https://data.mendeley.com/datasets/wc2yz9485m/1"
DATASET_API_URL = f"https://api.data.mendeley.com/datasets/{DATASET_ID}?version={DATASET_VERSION}"
API_HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; QuickCartIndiaStudentProject/1.0)",
    "Accept": "application/vnd.mendeley-public-dataset.1+json, application/json",
}

def read_json_url(url, headers=None, timeout=60):
    request = urllib.request.Request(url, headers=headers or API_HEADERS)
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))

try:
    dataset_metadata = read_json_url(DATASET_API_URL)
    print("Mendeley dataset details received.")
except Exception as error:
    raise RuntimeError(
        "Could not retrieve the public Mendeley dataset details from the current API. "
        "Check that the dataset is still public and that the API is reachable. "
        f"URL tried: {DATASET_API_URL}. Original error: {error}"
    )

# The dataset details normally include its published files. If not, ask
# the dedicated public-files endpoint for the same dataset version.
dataset_files = dataset_metadata.get("files", []) if isinstance(dataset_metadata, dict) else []
if not dataset_files:
    files_url = (
        f"https://api.data.mendeley.com/datasets/{DATASET_ID}/files"
        f"?version={DATASET_VERSION}&$limit=100"
    )
    try:
        files_response = read_json_url(files_url)
        if isinstance(files_response, list):
            dataset_files = files_response
        elif isinstance(files_response, dict):
            dataset_files = files_response.get("results", files_response.get("files", []))
    except Exception as error:
        raise RuntimeError(
            "Dataset details were received, but the public file list could not be read. "
            f"Files URL tried: {files_url}. Original error: {error}"
        )

# Select the CSV by its published filename, not by guessing a download URL.
csv_candidates = [
    item for item in dataset_files
    if isinstance(item, dict)
    and str(item.get("filename", item.get("name", ""))).lower().endswith(".csv")
]
if not csv_candidates:
    available_names = [
        str(item.get("filename", item.get("name", "unnamed file")))
        for item in dataset_files if isinstance(item, dict)
    ]
    raise RuntimeError(
        "The public dataset response did not contain a CSV file. "
        f"Files reported by Mendeley: {available_names}. "
        "Open DATASET_PAGE and check whether the publisher changed the files."
    )

# Prefer a file whose name looks like the quick-commerce survey.
csv_file = next(
    (item for item in csv_candidates
     if "quick" in str(item.get("filename", item.get("name", ""))).lower()
     or "survey" in str(item.get("filename", item.get("name", ""))).lower()),
    csv_candidates[0],
)
csv_file_name = csv_file.get("filename", csv_file.get("name", "dataset.csv"))
csv_file_id = csv_file.get("id") or csv_file.get("file_id")

if not csv_file_id:
    raise RuntimeError(
        f"Mendeley listed {csv_file_name}, but did not provide a file ID. "
        "Inspect dataset_metadata['files'] and update the file selection."
    )

# This public endpoint redirects to the actual file download.
download_url = (
    f"https://api.data.mendeley.com/datasets/{DATASET_ID}/files/"
    f"{csv_file_id}/file_downloaded?version={DATASET_VERSION}"
)
download_request = urllib.request.Request(
    download_url,
    headers={"User-Agent": API_HEADERS["User-Agent"]},
)

try:
    with urllib.request.urlopen(download_request, timeout=120) as response:
        downloaded_bytes = response.read()
        final_download_url = response.geturl()
except Exception as error:
    raise RuntimeError(
        f"Mendeley listed the CSV ({csv_file_name}), but its download failed. "
        f"Download endpoint: {download_url}. Original error: {error}"
    )

# Some dataset versions may deliver a ZIP archive instead of a plain CSV.
if downloaded_bytes[:2] == b"PK":
    with zipfile.ZipFile(io.BytesIO(downloaded_bytes)) as archive:
        csv_inside_zip = [
            name for name in archive.namelist()
            if name.lower().endswith(".csv") and not name.endswith("/")
        ]
        if not csv_inside_zip:
            raise RuntimeError(
                "Mendeley returned a ZIP archive, but no CSV was found inside it. "
                f"Archive contents: {archive.namelist()}"
            )
        preferred_csv = next(
            (name for name in csv_inside_zip
             if "quick" in name.lower() or "survey" in name.lower()),
            csv_inside_zip[0],
        )
        downloaded_bytes = archive.read(preferred_csv)
        csv_file_name = preferred_csv

raw_df = pd.read_csv(io.BytesIO(downloaded_bytes))
print("CSV downloaded and read from memory.")
print("Published file:", csv_file_name)
print("Downloaded size:", len(downloaded_bytes), "bytes")
print("Dataset shape:", raw_df.shape)
display(raw_df.head())


## 3. Check whether the downloaded file is the dataset I expect

I should not start analysis before checking that the correct file was downloaded. The original version used in this project contains 800 survey responses and 10 columns. If the source changes and the downloaded file has a different structure, I need to inspect that change rather than continue as though nothing happened.

I am checking the number of rows, number of columns and the column names. I am also keeping a copy of the original data in `raw_df`. I will not overwrite it during cleaning.

In [ ]:
EXPECTED_COLUMNS = [
    "Most_Used_App",
    "Ordering_Frequency_Change",
    "Average_Order_Value",
    "Reliance_On_Traditional_Bulk_Shopping",
    "Shifted_Categories",
    "Impulse_Buying_Frequency",
    "Impulse_Triggers",
    "No_Home_Inventory_Agreement",
    "Monthly_Budget_Percentage_On_Apps",
    "Overall_Grocery_Expenditure_Impact",
]

print("Rows:", raw_df.shape[0])
print("Columns:", raw_df.shape[1])
print("Column names:")
for number, column in enumerate(raw_df.columns, start=1):
    print(f"{number:02d}. {column}")

missing_columns = sorted(set(EXPECTED_COLUMNS) - set(raw_df.columns))
extra_columns = sorted(set(raw_df.columns) - set(EXPECTED_COLUMNS))
print("\nMissing expected columns:", missing_columns)
print("Additional columns:", extra_columns)

if missing_columns:
    raise ValueError("The downloaded file does not match the expected QuickCart survey columns.")

assert raw_df.shape[0] > 0, "The downloaded dataset is empty."

## 4. Understand what one row and each column mean

One row in this file is one person's survey response. It is not one order, invoice or real shopping transaction. The file does not contain the actual date of an order, the exact amount paid, a customer ID or a city name.

I am writing down the meaning of the columns before choosing a model. This matters because I should not use a column as an exact numerical value when it only stores a range or a written answer.

| Column | What I understand from it | How it is stored |
|---|---|---|
| `Most_Used_App` | Quick-commerce app used most often | Category |
| `Ordering_Frequency_Change` | How the person's ordering frequency changed | Category |
| `Average_Order_Value` | Approximate order-value band selected by the respondent | Category/range |
| `Reliance_On_Traditional_Bulk_Shopping` | Change in traditional monthly bulk shopping | Category |
| `Shifted_Categories` | Grocery categories moved to quick-commerce apps; multiple choices can be joined | Multi-select text |
| `Impulse_Buying_Frequency` | How often the respondent reports impulse buying | Ordered 1–5 answer |
| `Impulse_Triggers` | Things that encourage an impulse purchase; multiple choices can be joined | Multi-select text |
| `No_Home_Inventory_Agreement` | Agreement with the statement about ordering when items are not available at home | Ordered 1–5 answer |
| `Monthly_Budget_Percentage_On_Apps` | Share of monthly grocery budget spent through apps | Category/range |
| `Overall_Grocery_Expenditure_Impact` | Respondent's view of how quick commerce affects grocery spending | Category |

The AOV field is not exact money. For example, `Rs.250 to 500` does not tell me whether the order was ₹260 or ₹490. So I will keep it as a category.

In [ ]:
display(pd.DataFrame({
    "Column": raw_df.columns,
    "Data type": [str(raw_df[c].dtype) for c in raw_df.columns],
    "Different answers": [raw_df[c].nunique(dropna=True) for c in raw_df.columns],
    "Blank answers": [raw_df[c].isna().sum() for c in raw_df.columns],
}))

print("\nFull data information:")
raw_df.info()

## 5. First data-quality check — missing answers and repeated rows

A missing answer and a written answer such as `None` are not always the same thing. A blank app response means the person did not provide that answer. But the text `None (I still buy all categories in monthly store visits)` is a real answer to the grocery-category question.

I am checking actual missing values, empty text, duplicate rows and the text `None` separately. I will not delete rows just because one app answer is missing. I can keep those responses by using a clear category called `Not specified` for that app field.

In [ ]:
quality_check = pd.DataFrame({
    "Missing values": raw_df.isna().sum(),
    "Blank or whitespace": raw_df.apply(
        lambda column: column.astype("string").str.strip().eq("").sum()
    ),
    "Unique non-missing answers": raw_df.nunique(dropna=True),
})
display(quality_check)

print("Repeated full rows:", int(raw_df.duplicated().sum()))
print("Rows containing the literal word None in Shifted_Categories:",
      int(raw_df["Shifted_Categories"].astype("string").str.startswith("None", na=False).sum()))

## 6. Understand the target before selecting it

The target I am choosing is `Impulse_Buying_Frequency`. It has five ordered answers. The number at the beginning of each answer gives me a score from 1 to 5.

| Score | Meaning |
|---:|---|
| 1 | Never |
| 2 | Rarely |
| 3 | Sometimes |
| 4 | Frequently |
| 5 | Almost Always |

I am choosing this target because it is the clearest ordered numerical response already available in the survey. I am not choosing `Average_Order_Value` as the target because it contains unequal ranges and an open-ended `Above Rs.1000` group. Turning those bands into exact rupee amounts would add values that the survey never recorded.

There is one limitation: the values 1 to 5 are ordered choices, not measurements where the gap between every pair is guaranteed to be equal. Regression treats those gaps as approximately equal. I will mention this whenever I interpret the results.

In [ ]:
target_counts = raw_df["Impulse_Buying_Frequency"].value_counts(dropna=False)
display(target_counts.rename("Responses").to_frame())

print("Target is based on these original survey answers:")
for answer in sorted(raw_df["Impulse_Buying_Frequency"].dropna().unique()):
    print(answer)

## 7. Clean the dataset

I am now making a separate copy called `clean_df`. The original `raw_df` stays unchanged.

These are the changes I am making:

1. Fill the missing `Most_Used_App` answers with `Not specified`. I am not guessing which app they used.
2. Remove extra spaces at the start or end of text answers. This prevents the same answer from being treated as two different answers because of a space.
3. Extract the first number from the two 1–5 survey answers. I am adding two new score columns and keeping the original written answers too.
4. Check that the target scores are between 1 and 5.

I am not deleting unusual but valid answers, and I am not changing the ranges into made-up rupee values.

In [ ]:
clean_df = raw_df.copy()

# Keep missing app responses as a separate, visible answer.
clean_df["Most_Used_App"] = clean_df["Most_Used_App"].fillna("Not specified")

# Remove spaces around text answers without changing the answer itself.
for column in clean_df.select_dtypes(include=["object", "string"]).columns:
    clean_df[column] = clean_df[column].map(
        lambda value: value.strip() if isinstance(value, str) else value
    )

# Read the leading 1–5 number into new numeric columns.
def get_leading_score(series):
    leading_number = series.astype("string").str.extract(r"^\s*([1-5])", expand=False)
    return pd.to_numeric(leading_number, errors="coerce")

clean_df["Impulse_Buying_Frequency_Score"] = get_leading_score(
    clean_df["Impulse_Buying_Frequency"]
)
clean_df["No_Home_Inventory_Agreement_Score"] = get_leading_score(
    clean_df["No_Home_Inventory_Agreement"]
)

print("Cleaned data shape:", clean_df.shape)
print("Missing values after cleaning:")
display(clean_df.isna().sum().to_frame("Missing count"))
print("App answers marked Not specified:",
      int(clean_df["Most_Used_App"].eq("Not specified").sum()))
print("Target score values:", sorted(clean_df["Impulse_Buying_Frequency_Score"].dropna().unique()))

display(clean_df.head())

## 8. Final checks before saving the cleaned CSV

I am checking that the cleaning did what I expected. The original version of this dataset has 800 responses, 10 original columns, and 35 missing app answers. After adding two score columns, the cleaned data should have 12 columns.

I also check whether any score could not be extracted. If this check fails, I should inspect the original answer text instead of silently removing those rows.

In [ ]:
assert len(clean_df) == len(raw_df), "Rows should not disappear during this cleaning step."
assert clean_df["Most_Used_App"].isna().sum() == 0
assert clean_df["Impulse_Buying_Frequency_Score"].notna().all()
assert clean_df["No_Home_Inventory_Agreement_Score"].notna().all()
assert clean_df["Impulse_Buying_Frequency_Score"].between(1, 5).all()
assert clean_df["No_Home_Inventory_Agreement_Score"].between(1, 5).all()

print("All cleaning checks passed.")
print("Original rows:", len(raw_df))
print("Cleaned rows:", len(clean_df))
print("Cleaned columns:", len(clean_df.columns))

## 9. Download the cleaned dataset to my computer

I want the cleaned CSV to be available inside my project folder as well. This notebook will download it to the Colab session's download area. I will then place it manually in the `data` folder of my project.

The file name will be:

`quick_commerce_survey_responses_cleaned.csv`

The CSV is generated from the same cleaning steps above. I am not asking Colab to save it directly into a GitHub repository.

In [ ]:
CLEANED_CSV_NAME = "quick_commerce_survey_responses_cleaned.csv"
clean_df.to_csv(CLEANED_CSV_NAME, index=False, encoding="utf-8")
print("Cleaned dataset is ready:", CLEANED_CSV_NAME)

try:
    from google.colab import files
    files.download(CLEANED_CSV_NAME)
except ImportError:
    print("This notebook is not running in Google Colab. The file is saved in the current folder.")

## 10. Exploratory Data Analysis — why I am doing it

EDA means looking carefully at the data before training a model. I am using it to understand what answers are common, how responses are spread, whether some groups are very small, and whether there are visible relationships worth checking.

A graph does not automatically prove that one thing causes another. These graphs show what is present in these 800 survey responses only. They should not be treated as the exact behaviour of every quick-commerce customer in India.

### Graph 1 — Missing values in each original column

I am using a bar chart because it makes it easy to compare the number of blank answers from one column to another. The height of each bar represents how many answers are missing.

**What I should notice:** if one column has many missing values, I need to decide whether those rows can be kept, filled with a clear label or removed. In this dataset the main missing values are in `Most_Used_App`, and I have kept them as `Not specified`.

In [ ]:
missing_plot = raw_df.isna().sum().sort_values(ascending=False)
plt.figure(figsize=(11, 5))
missing_plot.plot(kind="bar")
plt.title("Missing answers by column")
plt.xlabel("Survey column")
plt.ylabel("Number of missing answers")
plt.xticks(rotation=65, ha="right")
plt.tight_layout()
plt.show()
print("Highest missing count:", int(missing_plot.max()))

### Graph 2 — Target score distribution

This bar chart shows how many respondents selected each impulse-buying frequency score. The five bars represent Never, Rarely, Sometimes, Frequently and Almost Always.

**Why I am using it:** before training, I need to know whether all five answer groups have enough responses. If one group is very small, predictions for that group may be less reliable. I will also use stratified splitting so the five score groups keep roughly the same proportions in both training and testing data.

In [ ]:
score_labels = {1: "Never", 2: "Rarely", 3: "Sometimes", 4: "Frequently", 5: "Almost Always"}
score_counts = clean_df["Impulse_Buying_Frequency_Score"].value_counts().reindex([1,2,3,4,5], fill_value=0)
plt.figure(figsize=(8, 5))
sns.barplot(x=[score_labels[i] for i in score_counts.index], y=score_counts.values)
plt.title("How often respondents report impulse buying")
plt.xlabel("Reported frequency")
plt.ylabel("Number of responses")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()
display(score_counts.rename("Responses").to_frame())

### Graph 3 — Most-used quick-commerce app

This chart compares the number of responses for each app. I am including `Not specified` as its own group so that the missing answers do not disappear from the picture.

**What it represents:** the bar height is the number of survey respondents who selected that app. It is not market share, app order volume or the number of active users in India. It only describes this survey sample.

In [ ]:
app_counts = clean_df["Most_Used_App"].value_counts()
plt.figure(figsize=(9, 5))
sns.barplot(x=app_counts.values, y=app_counts.index)
plt.title("Most-used quick-commerce app in this survey")
plt.xlabel("Number of responses")
plt.ylabel("App")
plt.tight_layout()
plt.show()
display(app_counts.rename("Responses").to_frame())

### Graph 4 — Ordering frequency change

This chart shows how respondents say their ordering frequency changed. I am using a horizontal bar chart because some answer labels are long.

**Why it is useful:** it helps me see whether most people report increasing, decreasing or maintaining their app orders. It is still a person's survey response, not a comparison of actual order histories.

In [ ]:
frequency_counts = clean_df["Ordering_Frequency_Change"].value_counts()
plt.figure(figsize=(10, 5))
sns.barplot(x=frequency_counts.values, y=frequency_counts.index)
plt.title("Reported change in ordering frequency")
plt.xlabel("Number of responses")
plt.ylabel("Answer")
plt.tight_layout()
plt.show()

### Graph 5 — Average order value bands

The graph counts how many people selected each order-value range. I am not calculating the average rupee amount because the actual value inside each range is unknown and the last range has no upper limit.

**What it represents:** it shows the distribution of the selected bands. I will use this column as a category in the model, not as an exact numerical feature.

In [ ]:
aov_order = ["Under Rs.250", "Rs.250 to 500", "Rs.500-1000", "Above Rs.1000"]
aov_counts = clean_df["Average_Order_Value"].value_counts().reindex(aov_order, fill_value=0)
plt.figure(figsize=(8, 5))
sns.barplot(x=aov_counts.index, y=aov_counts.values)
plt.title("Reported average order-value bands")
plt.xlabel("Order-value band")
plt.ylabel("Number of responses")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()
display(aov_counts.rename("Responses").to_frame())

### Graph 6 — Monthly grocery budget spent on apps

This chart compares the four budget ranges in the survey. A larger bar means more respondents selected that range.

**Why I am checking it:** spending share may be related to the way people use quick-commerce apps. I can study the relationship, but I cannot say the budget range causes impulse buying from this chart alone.

In [ ]:
budget_order = ["Less than 10%", "10% to 30%", "30% to 60%", "More than 60%"]
budget_counts = clean_df["Monthly_Budget_Percentage_On_Apps"].value_counts().reindex(budget_order, fill_value=0)
plt.figure(figsize=(8, 5))
sns.barplot(x=budget_counts.index, y=budget_counts.values)
plt.title("Share of monthly grocery budget spent on apps")
plt.xlabel("Reported budget range")
plt.ylabel("Number of responses")
plt.tight_layout()
plt.show()

### Graph 7 — Home-inventory agreement score

The survey stores this answer as written text with a number from 1 to 5. I extracted that number into `No_Home_Inventory_Agreement_Score`.

This chart shows the count of each agreement score. I am using the score column because the order of the answers matters, but I still keep the original answer text for reference.

In [ ]:
inventory_counts = clean_df["No_Home_Inventory_Agreement_Score"].value_counts().reindex([1,2,3,4,5], fill_value=0)
plt.figure(figsize=(8, 5))
sns.barplot(x=inventory_counts.index.astype(str), y=inventory_counts.values)
plt.title("Agreement with the home-inventory statement")
plt.xlabel("Agreement score (1 = Strongly Disagree, 5 = Strongly Agree)")
plt.ylabel("Number of responses")
plt.tight_layout()
plt.show()

### Graph 8 — Multi-select grocery categories

`Shifted_Categories` can contain more than one answer separated by semicolons. Counting the full text combinations would create many different groups, even when they share the same grocery category.

I am splitting each response into its individual choices and counting how often each choice was selected. One respondent can contribute to more than one category, so percentages can add up to more than 100% if I calculate them using respondents as the denominator.

In [ ]:
def split_multi_answer(value):
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split(";") if part.strip()]

category_mentions = {}
for response in clean_df["Shifted_Categories"]:
    for option in split_multi_answer(response):
        category_mentions[option] = category_mentions.get(option, 0) + 1
category_counts = pd.Series(category_mentions).sort_values(ascending=False)

plt.figure(figsize=(11, 6))
sns.barplot(x=category_counts.values, y=category_counts.index)
plt.title("Grocery categories selected by respondents")
plt.xlabel("Number of mentions")
plt.ylabel("Grocery category")
plt.tight_layout()
plt.show()
display(category_counts.rename("Mentions").to_frame())

### Graph 9 — Multi-select impulse-buying triggers

The `Impulse_Triggers` column also contains multiple choices. I am counting each trigger separately so that the graph shows which triggers were mentioned most often.

**How to read it:** a higher bar means that trigger appeared in more survey responses. It does not prove the trigger caused someone to buy something. Also, I am not using this question as an input feature in the main model because it is very close to the impulse-buying target question. Using it could make the model appear better by relying on information closely tied to the answer I am trying to estimate.

In [ ]:
trigger_mentions = {}
for response in clean_df["Impulse_Triggers"]:
    for option in split_multi_answer(response):
        trigger_mentions[option] = trigger_mentions.get(option, 0) + 1
trigger_counts = pd.Series(trigger_mentions).sort_values(ascending=False)

plt.figure(figsize=(11, 6))
sns.barplot(x=trigger_counts.values, y=trigger_counts.index)
plt.title("Reported triggers for impulse buying")
plt.xlabel("Number of mentions")
plt.ylabel("Trigger")
plt.tight_layout()
plt.show()
display(trigger_counts.rename("Mentions").to_frame())

### Graph 10 — Impulse score across order-value bands

I am comparing the average reported impulse-buying score across the four AOV bands. A box plot is useful here because it shows the middle of the scores and how spread out the responses are within each band.

The line inside each box is the median. The box shows the middle half of the observations. The points outside the whiskers can be individual observations that are farther from the middle.

**Important:** this is a comparison of survey scores across groups. It does not show that a higher order-value band causes more impulse buying.

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=clean_df, x="Average_Order_Value", y="Impulse_Buying_Frequency_Score", order=aov_order)
plt.title("Impulse-buying score across order-value bands")
plt.xlabel("Reported order-value band")
plt.ylabel("Impulse-buying frequency score (1–5)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()
display(clean_df.groupby("Average_Order_Value", observed=True)["Impulse_Buying_Frequency_Score"].agg(
    ["count", "mean", "median", "std"]
).reindex(aov_order))

### Graph 11 — Impulse score across app choices

This box plot checks whether the score distributions look different across the selected apps. It can help me notice possible patterns, but the app groups are not equal in size and this is only a survey sample.

I am not going to conclude that one app makes people more impulsive just because its box is higher. Other differences between respondents may also explain the pattern.

In [ ]:
app_order = clean_df["Most_Used_App"].value_counts().index.tolist()
plt.figure(figsize=(10, 5))
sns.boxplot(data=clean_df, x="Most_Used_App", y="Impulse_Buying_Frequency_Score", order=app_order)
plt.title("Impulse-buying score across most-used apps")
plt.xlabel("Most-used app")
plt.ylabel("Impulse-buying frequency score (1–5)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### Graph 12 — Correlation between the two score columns

Correlation tells me whether two numerical columns tend to move together in this dataset. I am calculating it only for the two ordered survey score columns.

The value can range from -1 to +1. A value near +1 means they tend to increase together, a value near -1 means one tends to decrease when the other increases, and a value near zero means there is little straight-line relationship. Since these are ordered survey answers, I will treat this as a simple descriptive check, not a deep measurement of behaviour.

In [ ]:
score_corr = clean_df[[
    "Impulse_Buying_Frequency_Score",
    "No_Home_Inventory_Agreement_Score"
]].corr(method="spearman")
plt.figure(figsize=(6, 4))
sns.heatmap(score_corr, annot=True, cmap="vlag", vmin=-1, vmax=1, fmt=".2f")
plt.title("Relationship between the two ordered survey scores")
plt.tight_layout()
plt.show()
display(score_corr)

### EDA conclusion — what I learned before modelling

At this stage I have checked the missing answers, score distribution, app responses, budget bands, order-value bands, multi-select categories and some group comparisons. These graphs describe the survey responses; they do not establish cause and effect.

The target still remains `Impulse_Buying_Frequency_Score`. I will use the home-inventory agreement score and the other survey fields as inputs. I will keep `Impulse_Triggers` out of the main model because that question is closely linked to the target and might give an unfairly easy input.

## 11. Prepare the input features and target

I now need to separate the information I give to the model from the score I want it to estimate.

**Target (`y`):** `Impulse_Buying_Frequency_Score`.

**Input (`X`):**
- Most used app
- Change in ordering frequency
- Average order-value band
- Reliance on traditional bulk shopping
- Monthly grocery budget range spent on apps
- Overall grocery expenditure impact
- Home-inventory agreement score
- Individual grocery-category indicators created from `Shifted_Categories`

I am leaving out the original target text, the target score itself, and `Impulse_Triggers`. I am also leaving out the original home-inventory text because I already have its numeric score. The model must not receive the answer it is trying to predict as an input.

In [ ]:
# Split the multi-select grocery categories into separate yes/no columns.
category_lists = clean_df["Shifted_Categories"].apply(split_multi_answer)
all_category_options = sorted({option for choices in category_lists for option in choices})

category_matrix = pd.DataFrame(index=clean_df.index)
for number, option in enumerate(all_category_options, start=1):
    category_matrix[f"Shifted_Category_{number:02d}"] = category_lists.apply(
        lambda choices: int(option in choices)
    )

# Keep the target separate from every input feature.
y = clean_df["Impulse_Buying_Frequency_Score"].astype(int).copy()

categorical_features = [
    "Most_Used_App",
    "Ordering_Frequency_Change",
    "Average_Order_Value",
    "Reliance_On_Traditional_Bulk_Shopping",
    "Monthly_Budget_Percentage_On_Apps",
    "Overall_Grocery_Expenditure_Impact",
]
numerical_features = ["No_Home_Inventory_Agreement_Score"]
binary_features = list(category_matrix.columns)

X = pd.concat(
    [clean_df[categorical_features + numerical_features].copy(), category_matrix],
    axis=1
)

print("Input shape:", X.shape)
print("Target shape:", y.shape)
print("Categorical columns:", categorical_features)
print("Numerical columns:", numerical_features)
print("Binary columns:", binary_features)
print("Target values:", sorted(y.unique()))
display(X.head())

## 12. Split into training and testing data

I am keeping 80% of the responses for training and 20% for testing.

- Training data is what the model learns from.
- Testing data is kept separate so I can check how the model behaves on responses it did not use while learning.

I am using `stratify=y`. This keeps the proportion of the five target-score groups approximately similar in both parts. I am using `random_state=42` so the same responses go into the same part when I run the notebook again.

I am doing the split before fitting the preprocessing steps. This is important because the test data must not influence how the model learns its categories or numerical scaling.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training target share:")
display(y_train.value_counts(normalize=True).sort_index().rename("Share").to_frame())
print("Testing target share:")
display(y_test.value_counts(normalize=True).sort_index().rename("Share").to_frame())

assert len(X_train) + len(X_test) == len(X)
assert set(X_train.index).isdisjoint(set(X_test.index))

## 13. Set up preprocessing — what are the options and what am I choosing?

The model cannot directly understand written app names or answer descriptions. I need to convert them into numbers first.

There are different ways to do that:

- **One-hot encoding:** makes a separate 0/1 column for each category. I am choosing this for the text categories because it does not pretend that one app or answer is bigger than another.
- **Label encoding:** gives each category a number such as 0, 1, 2. I am not using it for these input categories because the model could treat those numbers as though they have an order.
- **Scaling:** brings numerical values onto a comparable scale. I am using `StandardScaler` for the 1–5 home-inventory score.
- **Passing binary columns through:** the grocery-category indicators are already 0 or 1, so they do not need another conversion.

I am using `handle_unknown="ignore"` in the one-hot encoder. If the Streamlit app receives a category that was not present during training, the encoder will not crash because of it.

The preprocessing is inside each model pipeline. This way it is learned from training data only and then applied to testing data.

In [ ]:
categorical_preparation = Pipeline(steps=[
    ("fill_missing", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore"))
])

numerical_preparation = Pipeline(steps=[
    ("fill_missing", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_preparation, categorical_features),
        ("numerical", numerical_preparation, numerical_features),
        ("binary", "passthrough", binary_features),
    ],
    remainder="drop"
)

print("Preprocessing plan is ready.")
print("Text categories -> one-hot encoding")
print("Home inventory score -> fill if needed + standard scale")
print("Grocery indicators -> keep as 0/1")

## 14. Train the models I want to compare

I am not going to look only at Polynomial Regression and assume it is the best because it sounds more advanced. I will compare it with simpler choices on the exact same training and testing split.

I am comparing four models:

1. **Mean baseline:** always predicts the average target score from the training data. This is the minimum reference I should try to beat.
2. **Linear Regression:** learns a straight-line relationship between the prepared input columns and the target.
3. **Polynomial Regression, degree 2:** creates squared terms and combinations between features before fitting Linear Regression. It can describe more complicated patterns, but it can also create too many columns and overfit.
4. **Ridge Polynomial Regression, degree 2:** also uses degree-2 polynomial features, but Ridge adds a penalty to very large model weights. This can reduce overfitting. I am testing it rather than assuming that regularisation will improve the result.

I am not using degree 3 or higher in the main comparison because the number of generated columns can grow very quickly. I am starting with degree 2 and checking the result first.

In [ ]:
models = {
    "Mean baseline": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", DummyRegressor(strategy="mean"))
    ]),
    "Linear Regression": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]),
    "Polynomial Regression (degree 2)": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
        ("model", LinearRegression())
    ]),
    "Ridge Polynomial Regression (degree 2)": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
        ("model", Ridge(alpha=10.0))
    ]),
}

fitted_models = {}
predictions = {}
for model_name, model_pipeline in models.items():
    model_pipeline.fit(X_train, y_train)
    fitted_models[model_name] = model_pipeline
    predictions[model_name] = model_pipeline.predict(X_test)
    print(model_name, "training completed")

## 15. Evaluate each model using the same measures

I am using more than one measure because a single number does not explain every part of the prediction errors.

- **MAE:** average size of the error. For example, MAE 0.8 means the prediction is off by about 0.8 score points on average. Lower is better.
- **MSE:** squares the errors before averaging. Large mistakes receive a bigger penalty. Lower is better.
- **RMSE:** square root of MSE. It is in the same score units as the target. Lower is better.
- **R²:** compares the model with a simple mean-based reference. A value near 1 is better; 0 means about the same as that reference; a negative value means worse than predicting the mean of the test values.
- **Median Absolute Error:** the middle absolute error. It is less affected by a few very large mistakes than MAE.
- **Explained Variance:** tells how much of the variation in the target is accounted for by the predictions. It is another diagnostic, not a replacement for the error measures.

I am measuring both training and testing results. If training results are much better than testing results, that can be a warning that the model has memorised patterns that do not carry over to new responses.

In [ ]:
def regression_scores(actual, predicted):
    mse = mean_squared_error(actual, predicted)
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(actual, predicted),
        "Median AE": median_absolute_error(actual, predicted),
        "Explained variance": explained_variance_score(actual, predicted),
    }

results = {}
for name, fitted in fitted_models.items():
    train_pred = fitted.predict(X_train)
    test_pred = fitted.predict(X_test)
    train_scores = regression_scores(y_train, train_pred)
    test_scores = regression_scores(y_test, test_pred)
    row = {f"Train {key}": value for key, value in train_scores.items()}
    row.update({f"Test {key}": value for key, value in test_scores.items()})
    results[name] = row

metrics_table = pd.DataFrame(results).T
metrics_table = metrics_table.sort_values("Test MAE")
display(metrics_table.round(4))

best_by_mae = metrics_table["Test MAE"].idxmin()
baseline_test_mae = metrics_table.loc["Mean baseline", "Test MAE"]
best_test_mae = metrics_table.loc[best_by_mae, "Test MAE"]
print("Lowest test MAE:", best_by_mae)
print("Does any trained regression beat the baseline on test MAE?",
      bool((metrics_table.drop(index="Mean baseline")["Test MAE"] < baseline_test_mae).any()))

## 16. Graph — compare the test errors

I am comparing test MAE and test RMSE for all four models. The bars show how far predictions are from the real survey scores on average. Lower bars are better.

I am using testing scores here because the main question is how well the model handles responses it did not train on. A model with a very low training error but a high testing error is not a good result for this project.

In [ ]:
error_plot_data = metrics_table[["Test MAE", "Test RMSE"]].reset_index(names="Model")
error_plot_long = error_plot_data.melt(id_vars="Model", var_name="Measure", value_name="Error")
plt.figure(figsize=(11, 6))
sns.barplot(data=error_plot_long, x="Model", y="Error", hue="Measure")
plt.title("Test error comparison — lower is better")
plt.xlabel("Model")
plt.ylabel("Error in survey score points")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 17. Graph — actual score against predicted score

This graph compares the real score with the score estimated by each model. If a point is close to the diagonal line, that prediction is close to the actual answer. Points far from the line have larger errors.

Regression gives decimal values even though the target is from 1 to 5. That is expected. I will keep the raw decimal predictions for evaluation. I will only round and limit the score when showing a simple 1–5 label in the app.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10), sharex=True, sharey=True)
for ax, (name, predicted) in zip(axes.ravel(), predictions.items()):
    ax.scatter(y_test, predicted, alpha=0.65)
    ax.plot([1, 5], [1, 5], linestyle="--")
    ax.set_title(name)
    ax.set_xlabel("Actual survey score")
    ax.set_ylabel("Predicted score")
    ax.set_xlim(0.8, 5.2)
    ax.set_ylim(0.8, 5.2)
plt.suptitle("Actual score vs predicted score", y=1.02)
plt.tight_layout()
plt.show()

## 18. Graph — residual analysis for Polynomial Regression

A residual is the actual score minus the predicted score.

`Residual = Actual score - Predicted score`

If residuals are spread around the zero line without a clear shape, that is generally more reassuring. A visible curve or pattern can mean the model has not captured some structure. A very wide spread means the errors are large.

I am checking the degree-2 Polynomial Regression model here. I am not expecting the plot to prove the model is good; it is another way to inspect the mistakes.

In [ ]:
poly_name = "Polynomial Regression (degree 2)"
poly_pred = predictions[poly_name]
poly_residuals = y_test.to_numpy() - poly_pred

plt.figure(figsize=(9, 5))
plt.scatter(poly_pred, poly_residuals, alpha=0.65)
plt.axhline(0, linestyle="--")
plt.title("Residual plot — Polynomial Regression")
plt.xlabel("Predicted score")
plt.ylabel("Residual (actual - predicted)")
plt.tight_layout()
plt.show()
print("Mean residual:", round(float(np.mean(poly_residuals)), 4))
print("Residual standard deviation:", round(float(np.std(poly_residuals)), 4))

## 19. Graph — distribution of residuals

This histogram shows how the Polynomial Regression errors are distributed. The horizontal axis is the size and direction of the error. The vertical axis shows how many test responses have errors in each range.

A narrow distribution near zero would mean that many predictions are close to the real answer. A wide distribution or long tails means that some predictions are far away. Because the score only ranges from 1 to 5, even a one-point error can matter.

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(poly_residuals, bins=15, kde=True)
plt.axvline(0, linestyle="--")
plt.title("Distribution of Polynomial Regression residuals")
plt.xlabel("Residual (actual - predicted)")
plt.ylabel("Number of test responses")
plt.tight_layout()
plt.show()

## 20. Graph — errors for each actual target score

I am checking the absolute error separately for the five actual score groups. A box plot shows the middle error, the spread and any unusually large errors for each group.

This is useful because an overall average can hide that the model performs well for common score groups but poorly for a less common group. I will use the same test observations for this graph.

In [ ]:
error_by_score = pd.DataFrame({
    "Actual score": y_test.to_numpy(),
    "Absolute error": np.abs(y_test.to_numpy() - poly_pred)
})
plt.figure(figsize=(8, 5))
sns.boxplot(data=error_by_score, x="Actual score", y="Absolute error")
plt.title("Polynomial Regression error by actual score")
plt.xlabel("Actual survey score")
plt.ylabel("Absolute prediction error")
plt.tight_layout()
plt.show()
display(error_by_score.groupby("Actual score")["Absolute error"].agg(
    ["count", "mean", "median", "max"]
))

## 21. Check the rounding behaviour for the 1–5 answer

The model returns a decimal score, while the original survey uses five answer choices. For display, I can round the decimal to the nearest integer and keep it between 1 and 5. This is only a display conversion; I must not use the rounded value to calculate MAE or RMSE.

I am also checking the percentage of exact rounded matches. This is not the primary regression measure, but it gives an easy view of how often the model lands on the same 1–5 option.

In [ ]:
def score_to_label(value):
    rounded = int(np.clip(np.floor(float(value) + 0.5), 1, 5))
    return rounded, score_labels[rounded]

poly_display_scores = np.array([score_to_label(value)[0] for value in poly_pred])
exact_rounded_match = float(np.mean(poly_display_scores == y_test.to_numpy()))
within_one_point = float(np.mean(np.abs(poly_display_scores - y_test.to_numpy()) <= 1))
print(f"Rounded exact match: {exact_rounded_match:.1%}")
print(f"Rounded prediction within one point: {within_one_point:.1%}")

rounded_confusion = pd.crosstab(
    pd.Series(y_test.to_numpy(), name="Actual score"),
    pd.Series(poly_display_scores, name="Rounded prediction"),
    rownames=["Actual score"], colnames=["Rounded prediction"], dropna=False
).reindex(index=[1,2,3,4,5], columns=[1,2,3,4,5], fill_value=0)
plt.figure(figsize=(7, 6))
sns.heatmap(rounded_confusion, annot=True, fmt="d", cmap="Blues")
plt.title("Rounded score comparison — Polynomial Regression")
plt.tight_layout()
plt.show()

## 22. Cross-validation on the training data

A single train/test split gives me one result. To check whether the training result depends too much on that one split, I am also using five-fold cross-validation on the training data.

The training data is divided into five parts. Each time, four parts are used for training and the remaining part is used for checking. This repeats until every part has been used for checking once.

I am using negative MAE because Scikit-learn follows a scoring convention where higher scores are better. I will multiply by -1 to show the normal MAE value, where lower is better. The test set remains untouched during this step.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, estimator in models.items():
    cv_result = cross_validate(
        estimator, X_train, y_train,
        cv=cv,
        scoring={"mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error", "r2": "r2"},
        return_train_score=False,
        n_jobs=-1
    )
    cv_rows.append({
        "Model": name,
        "Mean CV MAE": -cv_result["test_mae"].mean(),
        "Std CV MAE": cv_result["test_mae"].std(),
        "Mean CV RMSE": -cv_result["test_rmse"].mean(),
        "Mean CV R2": cv_result["test_r2"].mean(),
    })
cv_table = pd.DataFrame(cv_rows).set_index("Model").sort_values("Mean CV MAE")
display(cv_table.round(4))

## 23. Pick the model carefully — do not choose only by training score

The model with the lowest test MAE is the best model according to that one held-out error measure. But I also need to check its test RMSE and R², and compare it with the mean baseline.

If no regression model improves on the baseline, I should keep the baseline as the honest reference instead of announcing that Polynomial Regression is successful. The purpose of this project is to understand what the data supports, not to force a good-looking result.

In [ ]:
regression_names = [name for name in metrics_table.index if name != "Mean baseline"]
best_regression_name = metrics_table.loc[regression_names, "Test MAE"].idxmin()
best_regression_mae = float(metrics_table.loc[best_regression_name, "Test MAE"])
baseline_mae = float(metrics_table.loc["Mean baseline", "Test MAE"])

if best_regression_mae < baseline_mae:
    selected_model_name = best_regression_name
    selection_note = (
        f"{best_regression_name} has a lower test MAE than the mean baseline "
        "on this split. I will use it for the demonstration, while still showing the limitations."
    )
else:
    selected_model_name = "Mean baseline"
    selection_note = (
        "None of the regression models has a lower test MAE than the mean baseline. "
        "The evidence from this test split does not support deploying a more complex "
        "regression model as a better predictor."
    )

print("Best regression by test MAE:", best_regression_name)
print("Selected reference for the demo:", selected_model_name)
print(selection_note)

if selected_model_name == "Mean baseline":
    print("The interactive app will still allow model comparison, but it will clearly mark this limitation.")

## 24. Inspect some actual and predicted values

I am displaying a small sample of test responses with the actual score and the predictions from the models. The raw predictions can be decimal values and may fall slightly below 1 or above 5. I am keeping them unchanged here because clipping them would hide the original prediction error.

The app can round the prediction for a simple label, but the evaluation must use raw predictions.

In [ ]:
prediction_review = pd.DataFrame({"Actual score": y_test.to_numpy()})
for name, values in predictions.items():
    prediction_review[name] = values
prediction_review["Polynomial rounded score"] = poly_display_scores
prediction_review["Polynomial absolute error"] = np.abs(y_test.to_numpy() - poly_pred)
display(prediction_review.head(20).round(3))

## 25. Save the selected model as a Joblib file

I need one file which the Streamlit app can load later. I am saving the complete fitted pipeline, not just the final regression step. The pipeline contains the preprocessing steps too, so a new input can go through the same conversions used during training.

The file name will be:

`quickcart_regression_model.joblib`

After Colab downloads it, I will manually place it in the project's `model` folder. Joblib files should only be loaded from a trusted source because they can contain Python objects.

In [ ]:
import joblib

selected_model = fitted_models[selected_model_name]
model_artifact = {
    "project_name": "QuickCart India",
    "target_column": "Impulse_Buying_Frequency_Score",
    "target_description": "Reported impulse-buying frequency score from 1 to 5",
    "selected_model_name": selected_model_name,
    "selected_pipeline": selected_model,
    "all_models": fitted_models,
    "feature_columns": list(X.columns),
    "categorical_features": categorical_features,
    "numerical_features": numerical_features,
    "binary_features": binary_features,
    "category_options": all_category_options,
    "score_labels": score_labels,
    "training_mean": float(y_train.mean()),
    "random_state": RANDOM_STATE,
    "sklearn_version": sklearn.__version__,
    "created_utc": datetime.now(timezone.utc).isoformat(),
}

MODEL_FILE = "quickcart_regression_model.joblib"
joblib.dump(model_artifact, MODEL_FILE)
print("Model file created:", MODEL_FILE)
print("Selected model:", selected_model_name)

## 26. Save metrics.json for the reports folder

I am saving the important experiment details in a JSON file. This includes the dataset size, training/testing split, selected target, all model scores and a short interpretation of the outcome.

The file name will be:

`metrics.json`

I will manually place this file inside the project's `reports` folder after it downloads.

In [ ]:
metrics_output = {
    "project": "QuickCart India",
    "target": "Impulse_Buying_Frequency_Score",
    "target_meaning": "Self-reported ordered survey score from 1 (Never) to 5 (Almost Always)",
    "dataset": {
        "source_page": DATASET_PAGE,
        "rows": int(len(clean_df)),
        "original_columns": int(len(raw_df.columns)),
        "cleaned_columns": int(len(clean_df.columns)),
        "missing_app_answers_filled": int(clean_df["Most_Used_App"].eq("Not specified").sum()),
    },
    "split": {
        "test_size": 0.20,
        "random_state": RANDOM_STATE,
        "training_rows": int(len(X_train)),
        "testing_rows": int(len(X_test)),
        "stratified_by_target": True,
    },
    "features": {
        "input_count_before_encoding": int(X.shape[1]),
        "categorical": categorical_features,
        "numerical": numerical_features,
        "binary": binary_features,
        "excluded_impulse_triggers": True,
    },
    "selected_model": selected_model_name,
    "best_regression_by_test_mae": best_regression_name,
    "regression_beats_baseline_by_test_mae": bool(best_regression_mae < baseline_mae),
    "models": metrics_table.replace({np.nan: None}).to_dict(orient="index"),
    "cross_validation": cv_table.reset_index().to_dict(orient="records"),
    "rounded_polynomial_exact_match": exact_rounded_match,
    "rounded_polynomial_within_one_point": within_one_point,
    "interpretation": selection_note,
    "limitations": [
        "The target is an ordinal survey choice, not a continuous measurement.",
        "Each row is a survey response, not an order transaction.",
        "The dataset has no exact purchase amount, location, age, income or customer ID.",
        "Results describe this dataset and split only; they do not prove causation.",
        "A held-out test result does not guarantee future real-world performance.",
    ],
    "versions": {"python": __import__("sys").version.split()[0], "pandas": pd.__version__, "scikit_learn": sklearn.__version__},
}

METRICS_FILE = "metrics.json"
with open(METRICS_FILE, "w", encoding="utf-8") as file:
    json.dump(metrics_output, file, indent=2, ensure_ascii=False)
print("Metrics file created:", METRICS_FILE)

## 27. Save supporting reports

Apart from the model and metrics, I am saving two more helpful files:

- `model_comparison.csv` — a table with all training and testing measures for every model.
- `test_predictions.csv` — the actual score and each model's prediction for every test response.

These files help me explain what happened in the experiment rather than showing only the final model.

In [ ]:
metrics_table.to_csv("model_comparison.csv", index_label="Model")
prediction_review.to_csv("test_predictions.csv", index=False)

print("Supporting reports created:")
for file_name in ["model_comparison.csv", "test_predictions.csv"]:
    print("-", file_name)

## 28. Check the model using one new survey-style example

Before downloading the final files, I am checking that the saved pipeline can accept one new row with the same input columns. This example is only a test of the code path. It is not a real customer record.

The model should receive the same raw categories and numeric score columns used during training. The one-hot encoder and scaling steps inside the pipeline handle the conversions automatically.

In [ ]:
sample_input = X_test.iloc[[0]].copy()
sample_prediction = selected_model.predict(sample_input)[0]
rounded_score, rounded_label = score_to_label(sample_prediction)

print("Example input row:")
display(sample_input)
print("Raw estimated score:", round(float(sample_prediction), 3))
print("Displayed score:", rounded_score, "-", rounded_label)
print("This is an estimate of a survey score, not a guaranteed prediction of future purchases.")

## 29. Download all final files from Colab

These are the files I need to collect after the notebook finishes:

| Downloaded file | Where I will place it in my project |
|---|---|
| `quick_commerce_survey_responses_cleaned.csv` | `data/` |
| `quickcart_regression_model.joblib` | `model/` |
| `metrics.json` | `reports/` |
| `model_comparison.csv` | `reports/` |
| `test_predictions.csv` | `reports/` |

The original raw dataset is already included in the ZIP under `data/quick_commerce_survey_responses_raw.csv`. I do not need to download that file again from Colab.

I am triggering the downloads one by one to avoid trying to download multiple files at exactly the same time. After the browser downloads them, I need to manually move them into the folders shown above.

In [ ]:
FILES_TO_DOWNLOAD = [
    CLEANED_CSV_NAME,
    MODEL_FILE,
    METRICS_FILE,
    "model_comparison.csv",
    "test_predictions.csv",
]

try:
    from google.colab import files
    for file_name in FILES_TO_DOWNLOAD:
        print("Downloading:", file_name)
        files.download(file_name)
except ImportError:
    print("Not running in Google Colab. Files are saved in the current working directory:")
    for file_name in FILES_TO_DOWNLOAD:
        print("-", file_name)

## 30. Final project checks

I am checking the important items one last time: the model file exists, the metrics file can be opened, the model predicts one row, and every prediction has a finite numerical value.

The result is complete only when I have downloaded the files and placed them into the correct folders in my project. I will then run the Streamlit app locally and deploy it using the same files.

In [ ]:
# Load the saved artifact back from disk to confirm that it can be read.
loaded_artifact = joblib.load(MODEL_FILE)
loaded_pipeline = loaded_artifact["selected_pipeline"]

with open(METRICS_FILE, "r", encoding="utf-8") as file:
    loaded_metrics = json.load(file)

final_test_prediction = loaded_pipeline.predict(X_test.iloc[:5])
assert len(final_test_prediction) == 5
assert np.isfinite(final_test_prediction).all()
assert loaded_artifact["target_column"] == "Impulse_Buying_Frequency_Score"
assert loaded_metrics["dataset"]["rows"] == len(clean_df)

print("All final checks passed.")
print("Model:", MODEL_FILE)
print("Metrics:", METRICS_FILE)
print("Selected model:", loaded_artifact["selected_model_name"])
print("First five predictions:", np.round(final_test_prediction, 3))

## 31. What this experiment tells me

I have completed the full process from downloading and inspecting the data to cleaning it, making graphs, preparing the input, training different models and checking the results.

The most important thing is the comparison with the mean baseline. If Polynomial Regression gives a low error on training data but a higher error on testing data, that means it learned the training responses too closely and did not work well on new responses. Adding polynomial features does not automatically make a model better.

This dataset is useful for practising the full regression workflow, but it has limits. It contains survey answers rather than real order records, and the target is an ordered 1–5 answer. I should explain those limits in the report and in the Streamlit app. I should not claim the model predicts exact impulse purchases, sales, or rupee amounts.

**My conclusion should come from the test results printed above.** The model I show in the web app should be described as an educational estimate based on survey patterns, not as a perfect or production-ready customer-behaviour predictor.